In [1]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.chat_history import InMemoryChatMessageHistory
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain_core.output_parsers import StrOutputParser
import os


llm = ChatGoogleGenerativeAI(
    model="gemini-3.5-flash",
    api_key=os.environ['GEMINI_API_KEY'],
    temperature=0
)


prompt = ChatPromptTemplate.from_messages([
    ("system", "You are a helpful assistant."),
    MessagesPlaceholder(variable_name="history"),
    ("human", "{input}")
])

chain = prompt | llm | StrOutputParser()

c:\Users\LENOVO\AppData\Local\Programs\Python\Python314\Lib\site-packages\langchain_core\utils\pydantic.py:41: UserWarning: Core Pydantic V1 functionality isn't compatible with Python 3.14 or greater.
  from pydantic.v1 import BaseModel as BaseModelV1
c:\Users\LENOVO\AppData\Local\Programs\Python\Python314\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
store = {}

def get_session_history(session_id):
    if session_id not in store:
        store[session_id] = InMemoryChatMessageHistory()
    return store[session_id]


chain_with_memory = RunnableWithMessageHistory(
    chain,
    get_session_history,
    input_messages_key="input",
    history_messages_key="history"
)

c:\Users\LENOVO\AppData\Local\Programs\Python\Python314\Lib\site-packages\IPython\core\interactiveshell.py:3701: LangChainDeprecationWarning: RunnableWithMessageHistory is deprecated. Use LangGraph's built-in persistence instead.
  exec(code_obj, self.user_global_ns, self.user_ns)


In [3]:
user1 = chain_with_memory.invoke(
    {"input": "My name is Alice."},
    config={"configurable": {"session_id": "user_1"}}
)

user2 = chain_with_memory.invoke(
    {"input": "My name is Bob."},
    config={"configurable": {"session_id": "user_2"}}
)

user3 = chain_with_memory.invoke(
    {"input": "My name is Charlie."},
    config={"configurable": {"session_id": "user_3"}}
)

print(user1)
print(user2)
print(user3)

Hello Alice! It's nice to meet you. How can I help you today?
Hello Bob! Nice to meet you. How can I help you today?
Hello Charlie! It's great to meet you. How can I help you today?


In [4]:
print(store)

{'user_1': InMemoryChatMessageHistory(messages=[HumanMessage(content='My name is Alice.', additional_kwargs={}, response_metadata={}), AIMessage(content="Hello Alice! It's nice to meet you. How can I help you today?", additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[])]), 'user_2': InMemoryChatMessageHistory(messages=[HumanMessage(content='My name is Bob.', additional_kwargs={}, response_metadata={}), AIMessage(content='Hello Bob! Nice to meet you. How can I help you today?', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[])]), 'user_3': InMemoryChatMessageHistory(messages=[HumanMessage(content='My name is Charlie.', additional_kwargs={}, response_metadata={}), AIMessage(content="Hello Charlie! It's great to meet you. How can I help you today?", additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[])])}


In [5]:
print(store["user_1"].messages)
print(store["user_2"].messages)
print(store["user_3"].messages)

[HumanMessage(content='My name is Alice.', additional_kwargs={}, response_metadata={}), AIMessage(content="Hello Alice! It's nice to meet you. How can I help you today?", additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[])]
[HumanMessage(content='My name is Bob.', additional_kwargs={}, response_metadata={}), AIMessage(content='Hello Bob! Nice to meet you. How can I help you today?', additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[])]
[HumanMessage(content='My name is Charlie.', additional_kwargs={}, response_metadata={}), AIMessage(content="Hello Charlie! It's great to meet you. How can I help you today?", additional_kwargs={}, response_metadata={}, tool_calls=[], invalid_tool_calls=[])]


In [6]:
user1_again = chain_with_memory.invoke(
    {"input": "What is my name?"},
    config={"configurable": {"session_id": "user_1"}}
)

print(user1_again)

Your name is Alice. How can I help you today, Alice?


In [7]:
different_user = chain_with_memory.invoke(
    {"input": "What is my name?"},
    config={"configurable": {"session_id": "user_4"}}
)

print(different_user)

I don't know your name yet! Since I don't have access to your personal information, you'll have to tell me. What should I call you?
